 classification binaire/multiclasse (statut, observation)

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

class FraudDetectionModel:
    def __init__(self):
        self.models_status = []
        self.models_observation = []
        self.label_encoders = {}
        self.scaler = StandardScaler()
        self.feature_columns = None
        self.class_weights_status = None
        self.class_weights_observation = None
        
    def prepare_data(self, df):
        print("\n""Préparation des données""\n")
        df = df.copy()
        print (df.columns)
        
        # Identifier les colonnes cibles
        target_status = 'status_verification'
        target_observation = 'observations_verification'
        
        # Encoder les cibles
        self.label_encoders[target_status] = LabelEncoder()
        df[f'{target_status}_encoded'] = self.label_encoders[target_status].fit_transform(df[target_status])
        
        # Pour observations_verification, créer une variable binaire (RAS vs autres)
        df[f'{target_observation}_binary'] = df[target_observation].apply(
            lambda x: 1 if str(x).strip() == 'RAS' else 0
        )
        
        # Encoder la colonne type_observation (avec fallback si manquante)
        le_type_observation = LabelEncoder()
        if 'type_observation' in df.columns:
            df['type_observation_encoded'] = le_type_observation.fit_transform(
                df['type_observation'].astype(str).fillna('missing')
            )
            df = df.drop('type_observation', axis=1)
        else:
            df['type_observation_encoded'] = 0
        
        
        # Sélectionner les features (exclure les colonnes cibles et inutiles)
        exclude_cols = [
            'status_verification', 'observations_verification',
            'status_verification_encoded', 'observations_verification_binary',
            
        ]
        
        # Identifier les colonnes de features
        feature_cols = [col for col in df.columns if col not in exclude_cols]
        print("\n les features retenues sont :\n",feature_cols)
        
        # Séparer les données
        X = df[feature_cols].fillna(0)
        y_status = df[f'{target_status}_encoded']
        y_observation = df[f'{target_observation}_binary']
        
        # Standardiser les features numériques
        numeric_cols = X.select_dtypes(include=[np.number]).columns
        X[numeric_cols] = self.scaler.fit_transform(X[numeric_cols])
        
        self.feature_columns = feature_cols
        
        return X, y_status, y_observation
    
    def calculate_class_weights(self, y):
        print("\n Calculer les poids des classes pour gérer le déséquilibre ""\n ")
        unique, counts = np.unique(y, return_counts=True)
        weights = len(y) / (len(unique) * counts)
        return weights / weights.sum()
    
    def create_lgb_dataset(self, X, y, sample_weight=None):
        print("\n ""Créer un dataset LightGBM ""\n ")
        return lgb.Dataset(
            X.values if hasattr(X, 'values') else X,
            label=y,
            weight=sample_weight,
            feature_name=list(X.columns) if hasattr(X, 'columns') else None
        )
    
    def train_models(self, X, y_status, y_observation, n_folds=5):
        print("""\n " ""Entraîner les modèles avec validation croisée" ""\n """)
        
        # Calculer les poids des classes
        self.class_weights_status = self.calculate_class_weights(y_status)
        self.class_weights_observation = self.calculate_class_weights(y_observation)
        
        # Paramètres LightGBM pour la classification binaire/multiclasse
        lgb_params_status = {
            'objective': 'multiclass' if len(np.unique(y_status)) > 2 else 'binary',
            'num_class': len(np.unique(y_status)) if len(np.unique(y_status)) > 2 else None,
            'metric': 'multi_logloss' if len(np.unique(y_status)) > 2 else 'binary_logloss',
            'boosting_type': 'gbdt',
            'learning_rate': 0.05,
            'num_leaves': 31,
            'max_depth': -1,
            'min_child_samples': 20,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'reg_alpha': 0.1,
            'reg_lambda': 0.1,
            'n_jobs': -1,
            'verbose': -1,
            'random_state': 42
        }
        
        lgb_params_observation = {
            'objective': 'binary',
            'metric': 'binary_logloss',
            'boosting_type': 'gbdt',
            'learning_rate': 0.05,
            'num_leaves': 31,
            'max_depth': -1,
            'min_child_samples': 20,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'reg_alpha': 0.1,
            'reg_lambda': 0.1,
            'n_jobs': -1,
            'verbose': -1,
            'random_state': 42,
            'is_unbalance': True  # Pour gérer le déséquilibre des classes
        }
        
        # Validation croisée stratifiée
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
        
        # Entraîner pour status_verification
        print("""\n Entraînement des modèles pour 'status_verification'..." "\n """)
        self.models_status = []
        
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_status)):
            print(f"  Fold {fold + 1}/{n_folds}")
            
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y_status.iloc[train_idx], y_status.iloc[val_idx]
            
            # Créer les poids d'échantillonnage
            sample_weight = np.array([self.class_weights_status[label] for label in y_train])
            
            # Créer les datasets LightGBM
            train_data = self.create_lgb_dataset(X_train, y_train, sample_weight)
            val_data = self.create_lgb_dataset(X_val, y_val)
            
            # Entraîner le modèle
            model = lgb.train(
                lgb_params_status,
                train_data,
                valid_sets=[val_data],
                num_boost_round=1000,
                callbacks=[
                    lgb.early_stopping(50),
                    lgb.log_evaluation(50)
                ]
            )
            
            self.models_status.append(model)
        
        # Entraîner pour observations_verification
        print("\nEntraînement des modèles pour 'observations_verification'...\n")
        self.models_observation = []
        
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_observation)):
            print(f"  Fold {fold + 1}/{n_folds}")
            
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y_observation.iloc[train_idx], y_observation.iloc[val_idx]
            
            # Créer les poids d'échantillonnage
            sample_weight = np.array([self.class_weights_observation[label] for label in y_train])
            
            # Créer les datasets LightGBM
            train_data = self.create_lgb_dataset(X_train, y_train, sample_weight)
            val_data = self.create_lgb_dataset(X_val, y_val)
            
            # Entraîner le modèle
            model = lgb.train(
                lgb_params_observation,
                train_data,
                valid_sets=[val_data],
                num_boost_round=1000,
                callbacks=[
                    lgb.early_stopping(50),
                    lgb.log_evaluation(50)
                ]
            )
            
            self.models_observation.append(model)
        
        print("\nEntraînement terminé!")
    
    def predict_proba(self, X):
        print("""\n ""Prédire les probabilités avec les modèles entraînés""")
        if not self.models_status or not self.models_observation:
            raise ValueError("Les modèles doivent être entraînés d'abord!")
        
        # Assurer que X a les mêmes colonnes que pendant l'entraînement
        X = pd.DataFrame(X, columns=self.feature_columns)
        
        # Standardiser les features
        numeric_cols = X.select_dtypes(include=[np.number]).columns
        X[numeric_cols] = self.scaler.transform(X[numeric_cols])
        
        # Prédictions pour status_verification (moyenne des folds)
        status_probs = []
        for model in self.models_status:
            prob = model.predict(X.values)
            status_probs.append(prob)
        
        status_avg_probs = np.mean(status_probs, axis=0)
        
        # Prédictions pour observations_verification (moyenne des folds)
        observation_probs = []
        for model in self.models_observation:
            prob = model.predict(X.values)
            observation_probs.append(prob)
        
        observation_avg_probs = np.mean(observation_probs, axis=0)
        
        return status_avg_probs, observation_avg_probs
    
    def predict(self, X, threshold_status=0.5, threshold_observation=0.5):
        print("""\n ""Prédire les classes avec seuils personnalisés """)
        status_probs, observation_probs = self.predict_proba(X)
        
        # Pour status_verification (multiclasse)
        if len(status_probs.shape) > 1 and status_probs.shape[1] > 1:
            status_pred = np.argmax(status_probs, axis=1)
        else:
            status_pred = (status_probs > threshold_status).astype(int)
        
        # Pour observations_verification (binaire)
        observation_pred = (observation_probs > threshold_observation).astype(int)
        
        # Décoder les prédictions si nécessaire
        if hasattr(self.label_encoders['status_verification'], 'inverse_transform'):
            status_pred_decoded = self.label_encoders['status_verification'].inverse_transform(status_pred)
        else:
            status_pred_decoded = status_pred
        
        return status_pred_decoded, observation_pred, status_probs, observation_probs
    
    def save_model(self, path):
        print("""\n ""Debut de Sauvegarde le modèle complet""")
        model_data = {
            'models_status': self.models_status,
            'models_observation': self.models_observation,
            'label_encoders': self.label_encoders,
            'scaler': self.scaler,
            'feature_columns': self.feature_columns,
            'class_weights_status': self.class_weights_status,
            'class_weights_observation': self.class_weights_observation
        }
        joblib.dump(model_data, path)
        print(f"Modèle sauvegardé à: {path}")
    
    def load_model(self, path):
        print("""\n ""Charger un modèle sauvegardé """)
        model_data = joblib.load(path)
        self.models_status = model_data['models_status']
        self.models_observation = model_data['models_observation']
        self.label_encoders = model_data['label_encoders']
        self.scaler = model_data['scaler']
        self.feature_columns = model_data['feature_columns']
        self.class_weights_status = model_data['class_weights_status']
        self.class_weights_observation = model_data['class_weights_observation']
        print(f"Modèle chargé depuis: {path}")

# Fonction pour tester avec une facture spécifique
def test_single_invoice(model, df, invoice_index=0):
    """Tester le modèle avec une facture spécifique"""
    print(f"\n{'='*60}")
    print(f"TEST AVEC UNE FACTURE (index: {invoice_index})")
    print(f"{'='*60}")
    
    # Préparer les données
    X, y_status, y_observation = model.prepare_data(df)
    
    # Sélectionner une facture
    single_invoice = X.iloc[[invoice_index]]
    true_status = y_status.iloc[invoice_index]
    true_observation = y_observation.iloc[invoice_index]
    
    # Faire une prédiction
    status_pred, observation_pred, status_probs, observation_probs = model.predict(single_invoice)
    
    # Afficher les résultats
    print(f"\nFacture testée (features sélectionnées):")
    print(f"Nombre de features: {len(single_invoice.columns)}")
    
    print(f"\nVérité terrain:")
    print(f"  Status vérification: {model.label_encoders['status_verification'].inverse_transform([true_status])[0]}")
    print(f"  Observations vérification: {'RAS' if true_observation == 1 else 'Autre'}")
    
    print(f"\nPrédictions du modèle:")
    print(f"  Status vérification prédit: {status_pred[0]}")
    print(f"  Observations vérification prédite: {'RAS' if observation_pred[0] == 1 else 'Autre'}")
    
    print(f"\nProbabilités:")
    if len(status_probs.shape) > 1:
        print(f"  Status - Probabilités par classe: {status_probs[0]}")
    else:
        print(f"  Status - Probabilité positive: {status_probs[0]:.4f}")
    print(f"  Observation - Probabilité RAS: {observation_probs[0]:.4f}")
    
    # Calculer la précision
    status_correct = 1 if status_pred[0] == model.label_encoders['status_verification'].inverse_transform([true_status])[0] else 0
    observation_correct = 1 if observation_pred[0] == true_observation else 0
    
    print(f"\nExactitude:")
    print(f"  Status vérification: {'CORRECT' if status_correct else 'INCORRECT'}")
    print(f"  Observations vérification: {'CORRECT' if observation_correct else 'INCORRECT'}")
    
    return {
        'invoice_data': single_invoice,
        'true_status': true_status,
        'true_observation': true_observation,
        'pred_status': status_pred[0],
        'pred_observation': observation_pred[0],
        'status_probs': status_probs[0],
        'observation_probs': observation_probs[0]
    }

# Exemple d'utilisation principale
if __name__ == "__main__":
    # Charger les données
    print("\nChargement des données...")
    df = pd.read_csv('dfforml.csv')
    
    print("\nInitialisation du modèle et préparation des données...")
    # Initialiser le modèle
    model = FraudDetectionModel()
    
    # Préparer les données
    X, y_status, y_observation = model.prepare_data(df)
    
    # Afficher des informations sur les données
    print(f"\nInformations sur le dataset:")
    print(f"Nombre d'échantillons: {len(df)}")
    print(f"Nombre de features: {len(X.columns)}")
    print(f"Classes status_verification: {np.unique(y_status)}")
    print(f"Distribution observations_verification: {np.unique(y_observation, return_counts=True)}")
    
    print(f"\nDiviser les données en ensembles d'entraînement et de test...")
    # Diviser les données
    X_train, X_test, y_status_train, y_status_test, y_observation_train, y_observation_test = train_test_split(
        X, y_status, y_observation, test_size=0.2, random_state=42, stratify=y_status
    )
    print(f"\nentraînement: {len(X_train)} échantillons, test: {len(X_test)} échantillons")
    # Entraîner le modèle
    model.train_models(X_train, y_status_train, y_observation_train, n_folds=3)
    
    # Évaluer sur l'ensemble de test
    print(f"\n{'='*60}")
    print("ÉVALUATION SUR L'ENSEMBLE DE TEST")
    print(f"{'='*60}")
    
    status_pred, observation_pred, status_probs, observation_probs = model.predict(X_test)
    
    # Pour status_verification
    print("\nPerformance pour 'status_verification':")
    print(classification_report(
        model.label_encoders['status_verification'].inverse_transform(y_status_test),
        status_pred
    ))
    
    # Pour observations_verification
    print("\nPerformance pour 'observations_verification':")
    print(classification_report(y_observation_test, observation_pred, 
                               target_names=['Autre', 'RAS']))
    
    # Calculer l'AUC ROC pour observations_verification
    if len(np.unique(y_observation_test)) > 1:
        auc_roc = roc_auc_score(y_observation_test, observation_probs)
        print(f"AUC ROC pour observations_verification: {auc_roc:.4f}")
    
    # Sauvegarder le modèle
    model.save_model('fraud_detection_model.pkl')
    
    # Tester avec une facture spécifique
    test_single_invoice(model, df, invoice_index=10)
    
    # Exemple de chargement et réutilisation du modèle
    print(f"\n{'='*60}")
    print("EXEMPLE DE RÉUTILISATION DU MODÈLE")
    print(f"{'='*60}")
    
    # Charger le modèle
    new_model = FraudDetectionModel()
    new_model.load_model('fraud_detection_model.pkl')
    
    # Tester avec une autre facture
    test_single_invoice(new_model, df, invoice_index=50)

In [ ]:
# Pour ajuster les hyperparamètres
custom_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.01,  # Plus petit pour plus de stabilité
    'num_leaves': 63,  # Plus grand pour plus de complexité
    'max_depth': 7,  # Limiter la profondeur
    'min_child_samples': 50,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.5,  # Régularisation L1
    'reg_lambda': 0.5,  # Régularisation L2
    'n_jobs': -1,
    'verbose': -1,
    'random_state': 42
}

# Pour utiliser XGBoost ou CatBoost à la place :
# Remplacez simplement l'import et l'API d'entraînement

Pour utiliser le modèle sur de nouvelles données :

In [ ]:
# Charger un modèle entraîné
model = FraudDetectionModel()
model.load_model('fraud_detection_model.pkl')

# Préparer de nouvelles données (doivent avoir les mêmes colonnes)
new_data = pd.read_csv('nouvelles_factures.csv')
X_new, _, _ = model.prepare_data(new_data)

# Faire des prédictions
status_pred, observation_pred, probs_status, probs_obs = model.predict(X_new)

# Ajuster les seuils si nécessaire
status_pred, observation_pred, _, _ = model.predict(
    X_new, 
    threshold_status=0.6,  # Seuil plus strict
    threshold_observation=0.4  # Seuil plus permissif
)